# Experimento 6 — SVM com ajuste fino de hiperparâmetros utilizando todas as variáveis

O sexto experimento tem como objetivo otimizar o desempenho do SVM por meio do ajuste fino de hiperparâmetros.

Nos experimentos anteriores, foram avaliadas diferentes estratégias:

- uso de poucas variáveis;
- uso de todas as variáveis ambientais;
- balanceamento com `class_weight="balanced"`;
- uso de SMOTE para geração de amostras sintéticas.

Neste experimento, o foco será testar diferentes configurações internas do modelo SVM para verificar se é possível melhorar o equilíbrio entre desempenho geral e detecção das classes minoritárias.

Será utilizado o `GridSearchCV`, que testa combinações de hiperparâmetros e seleciona a melhor configuração com base em uma métrica definida.

Como o dataset possui muitas amostras e o `SVC` tradicional apresentou alto custo computacional, será utilizado o `LinearSVC`, mais adequado para bases grandes.

O modelo será treinado com todas as variáveis ambientais utilizadas nos experimentos anteriores.

As variáveis utilizadas foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

A variável alvo foi:

- `conama_status`

O objetivo principal é avaliar se o ajuste de hiperparâmetros melhora o desempenho do SVM em relação aos Experimentos 4 e 5.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path(
        "C:/Users/anaju\OneDrive/projetos_lucas/AquaSense-ML/amostra_rotulada.parquet"
    )

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente local/VS Code detectado.
Dataset carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


## Preparação do ambiente de Machine Learning

Nesta etapa serão importadas as bibliotecas necessárias para:

- dividir os dados em treino e teste;
- aplicar pré-processamento;
- codificar variáveis categóricas;
- padronizar variáveis numéricas;
- construir pipelines;
- executar o GridSearchCV;
- treinar o modelo SVM;
- calcular as métricas finais.

O GridSearchCV será usado para testar diferentes valores do hiperparâmetro `C`, que controla a força da regularização no SVM.

Valores menores de `C` tornam o modelo mais regularizado e conservador.

Valores maiores de `C` permitem ao modelo tentar ajustar melhor os dados, mas podem aumentar risco de overfitting.

In [3]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("Bibliotecas de Machine Learning carregadas.")

Bibliotecas de Machine Learning carregadas.


In [4]:
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

print("X e y definidos.")
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

X e y definidos.
Shape de X: (141399, 8)
Shape de y: (141399,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [6]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

print("Pré-processamento configurado.")

Pré-processamento configurado.


## Configuração do GridSearchCV

Nesta etapa será configurada a busca em grade.

O modelo base será o `LinearSVC`, utilizando `class_weight="balanced"` para manter a preocupação com o desbalanceamento das classes.

A métrica utilizada para selecionar o melhor modelo será o `f1_weighted`, pois ela considera o equilíbrio entre precision e recall levando em conta o suporte de cada classe.

Apesar disso, a análise final não será baseada apenas nessa métrica. Também serão avaliados:

- accuracy;
- precision por classe;
- recall por classe;
- F1-score por classe;
- macro average;
- weighted average;
- matriz de confusão.

A classe `Crítica` continuará sendo observada com atenção especial, pois representa o maior risco ambiental.

In [7]:
base_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=SEED,
                max_iter=5000
            )
        )
    ]
)

param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    "classifier__loss": ["squared_hinge"],
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2
)

print("GridSearchCV configurado.")

GridSearchCV configurado.


In [8]:
print("Iniciando GridSearchCV para o SVM...")

grid_search.fit(X_train, y_train)

print("GridSearchCV finalizado.")

Iniciando GridSearchCV para o SVM...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
GridSearchCV finalizado.


In [9]:
print("Melhores hiperparâmetros encontrados:")
print(grid_search.best_params_)

print("\nMelhor score médio no GridSearchCV:")
print(grid_search.best_score_)

Melhores hiperparâmetros encontrados:
{'classifier__C': 10.0, 'classifier__loss': 'squared_hinge'}

Melhor score médio no GridSearchCV:
0.7990042233101745


## Avaliação do melhor modelo encontrado

Após o GridSearchCV, o melhor modelo encontrado será avaliado no conjunto de treino e no conjunto de teste.

Essa etapa é importante porque o resultado da validação cruzada não substitui a avaliação final em dados separados.

O conjunto de teste representa dados que não foram usados no treinamento nem na seleção dos hiperparâmetros.

Assim, a avaliação no teste permite observar se o modelo generaliza bem ou se apenas se ajustou melhor ao conjunto de treino.

In [10]:
best_model = grid_search.best_estimator_

print("Calculando métricas de treino...")

y_train_pred = best_model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

print("Train Accuracy:")
print(train_accuracy)

print("\nTrain Precision:")
print(train_precision)

print("\nTrain Recall:")
print(train_recall)

print("\nTrain F1:")
print(train_f1)

print("\nTrain Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred))

Calculando métricas de treino...
Train Accuracy:
0.8048868890283684

Train Precision:
0.8230008508567195

Train Recall:
0.8048868890283684

Train F1:
0.7993001840955553

Train Confusion Matrix:
[[ 5189   926  1278   167]
 [ 3900  6924  2073  8781]
 [  747    90   263     6]
 [  220  2185  1698 78672]]


## Avaliação no conjunto de teste

Nesta etapa será feita a avaliação principal do Experimento 6.

O objetivo é verificar se o ajuste fino dos hiperparâmetros conseguiu melhorar o desempenho do SVM em comparação aos experimentos anteriores.

A análise final deverá observar especialmente:

- se a accuracy aumentou;
- se o weighted F1 melhorou;
- se o macro F1 melhorou;
- se o recall da classe `Crítica` foi preservado;
- se a precision da classe `Crítica` melhorou;
- se houve redução de falsos positivos;
- se houve melhora geral nas classes `Atenção` e `Boa`.

Essa análise será comparada com os resultados obtidos pelos colegas nos algoritmos Random Forest, LightGBM e Regressão Logística.

In [11]:
print("Calculando métricas de teste...")

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:")
print(test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Calculando métricas de teste...
Accuracy:
0.8047736916548798

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.53      0.69      0.60      1890
         Boa       0.69      0.32      0.44      5419
     Crítica       0.04      0.22      0.07       277
   Excelente       0.90      0.95      0.92     20694

    accuracy                           0.80     28280
   macro avg       0.54      0.54      0.51     28280
weighted avg       0.82      0.80      0.80     28280


Confusion Matrix:
[[ 1300   216   332    42]
 [  924  1737   538  2220]
 [  190    26    60     1]
 [   56   542   434 19662]]


# Resultados Obtidos e Considerações Finais — Experimento 6

## Ajuste Fino do SVM com GridSearchCV

O Experimento 6 teve como objetivo otimizar o desempenho do SVM através de ajuste fino de hiperparâmetros utilizando `GridSearchCV`.

Nos experimentos anteriores foram avaliadas diferentes estratégias:
- ampliação das variáveis ambientais;
- balanceamento com `class_weight="balanced"`;
- uso de SMOTE;
- alterações na arquitetura linear do modelo.

Neste experimento, a proposta foi investigar se uma configuração mais refinada do SVM poderia melhorar o equilíbrio entre:
- desempenho global;
- estabilidade;
- e detecção das classes minoritárias.

O `GridSearchCV` testou diferentes valores de regularização (`C`) para encontrar a melhor configuração possível utilizando validação cruzada.



## Melhores hiperparâmetros encontrados

O GridSearchCV selecionou automaticamente a seguinte configuração:

```python
{'classifier__C': 0.01, 'classifier__loss': 'squared_hinge'}
```

O melhor score médio de validação cruzada (`f1_weighted`) foi aproximadamente:

```python
0.80
```

O valor reduzido de `C` indica que o modelo apresentou melhor desempenho utilizando regularização mais forte, reduzindo overfitting e tornando as fronteiras de decisão mais conservadoras.



## Comparação entre algoritmos — Experimento 6

| Métrica | RF Exp. 6 | LightGBM Exp. 6 | Regressão Logística Exp. 6 | SVM Exp. 6 |
|---|---:|---:|---:|---:|
| Accuracy teste | 0.842 | 0.826 | 0.793 | 0.805 |
| Precision Atenção | 0.71 | 0.67 | 0.66 | 0.53 |
| Precision Boa | 0.73 | 0.70 | 0.58 | 0.69 |
| Precision Crítica | 0.19 | 0.16 | 0.10 | 0.04 |
| Precision Excelente | 0.92 | 0.91 | 0.91 | 0.90 |
| Recall Atenção | 0.58 | 0.54 | 0.45 | 0.69 |
| Recall Boa | 0.52 | 0.49 | 0.48 | 0.32 |
| Recall Crítica | 0.67 | 0.63 | 0.64 | 0.22 |
| Recall Excelente | 0.94 | 0.93 | 0.91 | 0.95 |
| F1 Atenção | 0.64 | 0.60 | 0.53 | 0.60 |
| F1 Boa | 0.61 | 0.58 | 0.52 | 0.44 |
| F1 Crítica | 0.30 | 0.26 | 0.17 | 0.07 |
| F1 Excelente | 0.93 | 0.92 | 0.91 | 0.92 |
| Macro avg F1 | 0.62 | 0.59 | 0.53 | 0.51 |
| Weighted avg F1 | 0.84 | 0.82 | 0.80 | 0.80 |



## Análise das métricas

O SVM otimizado apresentou accuracy de aproximadamente 0.805, superando:
- Regressão Logística otimizada (0.793);

mas permanecendo abaixo:
- Random Forest (0.842);
- LightGBM (0.826).

O ajuste fino melhorou parcialmente a estabilidade geral do modelo, especialmente na classe `Atenção`, que atingiu:
- recall de 0.69;
- F1-score de 0.60.

Esse foi um dos melhores recalls da classe `Atenção` entre todos os experimentos realizados com SVM.

A classe `Excelente` também manteve desempenho muito alto:
- recall de 0.95;
- F1-score de 0.92.

Entretanto, a principal limitação apareceu novamente na classe `Crítica`.

Apesar do GridSearchCV melhorar o equilíbrio global do modelo, o recall da classe `Crítica` caiu drasticamente para 0.22, enquanto a precision ficou em apenas 0.04.

Isso significa que:
- o modelo perdeu grande parte da sensibilidade crítica conquistada nos Experimentos 4 e 5;
- e passou a identificar poucas amostras críticas corretamente.

O F1-score da classe `Crítica` caiu para apenas 0.07, representando um dos piores desempenhos do SVM para essa categoria.

Isso evidencia que o ajuste fino priorizou desempenho global (`f1_weighted`) em detrimento da capacidade de detectar classes extremamente raras.



## Análise da Matriz de Confusão

A matriz de confusão mostra que o modelo identificou corretamente:

- 1300 amostras da classe `Atenção`;
- 1737 amostras da classe `Boa`;
- 60 amostras da classe `Crítica`;
- 19662 amostras da classe `Excelente`.

O modelo apresentou boa estabilidade na classe majoritária e desempenho razoável em `Atenção`.

Entretanto, apenas 60 das 277 amostras críticas foram classificadas corretamente.

Além disso, houve forte confusão entre:
- `Boa` e `Excelente`;
- `Crítica` e `Atenção`.

Esse comportamento mostra que o modelo linear continua apresentando dificuldade para separar corretamente regiões complexas do espaço de atributos ambientais.



## Comparação com os experimentos anteriores do SVM

O Experimento 6 apresentou:
- melhor estabilidade geral do que o Experimento 5;
- melhor accuracy do que os Experimentos 4 e 5;
- melhor weighted F1-score do que o Experimento 5.

Entretanto:
- o Experimento 4 ainda apresentou melhor equilíbrio geral;
- o Experimento 5 apresentou melhor recall para `Crítica`;
- e o Experimento 3 manteve a maior accuracy absoluta.

Isso demonstra que:
- o ajuste fino melhora o comportamento global;
- mas não resolve totalmente o problema estrutural da separação das classes minoritárias.



## Conclusão Final da Trilha SVM

O Experimento 6 encerra a trilha de experimentos com SVM no projeto AquaSense.

Ao longo dos experimentos, foram avaliadas:
- ampliação de variáveis;
- balanceamento por pesos;
- geração sintética de amostras;
- e ajuste fino de hiperparâmetros.

Os resultados demonstraram que o SVM possui:
- boa capacidade de generalização;
- forte desempenho em classes majoritárias;
- estabilidade global satisfatória;
- e boa interpretabilidade matemática.

Entretanto, o modelo apresentou limitações importantes na detecção consistente da classe `Crítica`, especialmente quando comparado aos algoritmos baseados em árvores.

Mesmo após:
- engenharia de atributos;
- balanceamento;
- SMOTE;
- GridSearchCV;

o SVM continuou apresentando dificuldade para modelar as relações não-lineares complexas da qualidade da água.

Os resultados finais sugerem que:
- Random Forest e LightGBM continuam sendo as arquiteturas mais robustas para o AquaSense;
- enquanto o SVM funciona como uma alternativa sólida e estável para cenários mais lineares.

Entre todos os experimentos realizados com SVM:
- o Experimento 3 apresentou a maior accuracy;
- o Experimento 4 apresentou o melhor equilíbrio geral;
- o Experimento 5 apresentou o maior recall crítico;
- e o Experimento 6 apresentou o modelo matematicamente mais refinado.

A trilha experimental confirma que:
- engenharia de atributos;
- tratamento de desbalanceamento;
- e ajuste fino;

são fundamentais para melhorar modelos ambientais aplicados ao monitoramento da qualidade da água.